# 05 Service Performance Analysis

## Objective

Assess the available service-performance information in the customer support ticket dataset, including ticket status, timestamp coverage, timestamp ordering, the interval between first-response and resolution timestamps, and customer satisfaction.

**Important metric definition:** The dataset does not contain a ticket-created timestamp. Therefore, this notebook cannot calculate elapsed time from ticket creation to first response or total resolution duration from ticket creation. Where both timestamps are available, it calculates only the interval from the recorded first-response timestamp to the recorded resolution timestamp.

## Scope and limitations

- Do not treat timestamp strings as numeric values.
- Parse timestamp columns with `pd.to_datetime`.
- Investigate negative timestamp intervals separately; do not include them in valid interval summaries.
- Do not claim SLA breaches because no SLA target is supplied.
- Missing CSAT values are not zero ratings.
- Do not include customer names, email addresses, or ticket descriptions in outputs.

## 1. Load the dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/raw/customer_support_tickets.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place the CSV in data/raw/customer_support_tickets.csv."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Total tickets:", len(df))
print("Required columns present:",
      all(c in df.columns for c in [
          "Ticket ID", "Ticket Status", "First Response Time",
          "Time to Resolution", "Customer Satisfaction Rating"
      ]))

## 2. Validate records and ticket status

In [ ]:
required_columns = [
    "Ticket ID", "Ticket Status", "First Response Time",
    "Time to Resolution", "Customer Satisfaction Rating"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Duplicate Ticket IDs:", df["Ticket ID"].duplicated().sum())
print("Exact duplicate rows:", df.duplicated().sum())

status_summary = (
    df["Ticket Status"].fillna("Unknown").value_counts()
    .rename_axis("Ticket Status")
    .reset_index(name="Ticket Count")
)
status_summary["Percentage"] = (
    status_summary["Ticket Count"] / len(df) * 100
).round(2)

display(status_summary)

## 3. Parse timestamp fields correctly

In [ ]:
service_df = df.copy()

# These two columns contain date-time strings, not numeric durations.
service_df["first_response_dt"] = pd.to_datetime(
    service_df["First Response Time"], errors="coerce"
)
service_df["resolution_dt"] = pd.to_datetime(
    service_df["Time to Resolution"], errors="coerce"
)

timestamp_coverage = pd.DataFrame({
    "Field": ["First Response Time", "Time to Resolution"],
    "Available Records": [
        service_df["first_response_dt"].notna().sum(),
        service_df["resolution_dt"].notna().sum()
    ],
    "Missing / Invalid Records": [
        service_df["first_response_dt"].isna().sum(),
        service_df["resolution_dt"].isna().sum()
    ]
})
timestamp_coverage["Availability (%)"] = (
    timestamp_coverage["Available Records"] / len(service_df) * 100
).round(2)

display(timestamp_coverage)

## 4. Calculate the supported timestamp interval

In [ ]:
# This is the interval between two recorded timestamps only.
# It is NOT time from ticket creation to first response or total resolution time.

paired = service_df[
    service_df["first_response_dt"].notna()
    & service_df["resolution_dt"].notna()
].copy()

paired["first_response_to_resolution"] = (
    paired["resolution_dt"] - paired["first_response_dt"]
)

negative_intervals = paired[
    paired["first_response_to_resolution"] < pd.Timedelta(0)
].copy()

zero_intervals = paired[
    paired["first_response_to_resolution"] == pd.Timedelta(0)
].copy()

valid_intervals = paired[
    paired["first_response_to_resolution"] >= pd.Timedelta(0)
].copy()

print("Records with both timestamps:", len(paired))
print("Negative intervals:", len(negative_intervals))
print("Zero intervals:", len(zero_intervals))
print("Non-negative intervals:", len(valid_intervals))

if not negative_intervals.empty:
    print("\nExamples of negative timestamp sequences (Ticket ID only):")
    display(
        negative_intervals[[
            "Ticket ID", "first_response_dt", "resolution_dt",
            "first_response_to_resolution"
        ]].head(10)
    )

# Convert the valid interval to hours for readable descriptive statistics.
valid_intervals["interval_hours"] = (
    valid_intervals["first_response_to_resolution"].dt.total_seconds()
    / 3600
)

if not valid_intervals.empty:
    interval_summary = pd.DataFrame({
        "Metric": ["Valid first-response-to-resolution interval"],
        "Records": [len(valid_intervals)],
        "Median (hours)": [valid_intervals["interval_hours"].median()],
        "Mean (hours)": [valid_intervals["interval_hours"].mean()],
        "P90 (hours)": [valid_intervals["interval_hours"].quantile(0.90)],
        "Maximum (hours)": [valid_intervals["interval_hours"].max()]
    }).round(2)
    display(interval_summary)
else:
    print("No valid non-negative timestamp intervals are available.")

## 5. Visualise valid timestamp intervals

In [ ]:
if not valid_intervals.empty:
    plt.figure(figsize=(9, 5))
    plt.hist(valid_intervals["interval_hours"], bins=30)
    plt.title("Interval Between First Response and Resolution Timestamps")
    plt.xlabel("Interval (hours)")
    plt.ylabel("Number of Tickets")
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()
else:
    print("Chart not generated because no valid intervals are available.")

## 6. Customer satisfaction analysis

In [ ]:
csat = pd.to_numeric(
    service_df["Customer Satisfaction Rating"], errors="coerce"
)
rated = csat.dropna()

print("Tickets with CSAT:", len(rated))
print("Tickets without CSAT:", csat.isna().sum())
if not rated.empty:
    print("Average CSAT:", round(rated.mean(), 2))
    print("Median CSAT:", rated.median())
    print("Minimum CSAT:", rated.min())
    print("Maximum CSAT:", rated.max())

csat_distribution = (
    rated.value_counts().sort_index()
    .rename_axis("CSAT Rating")
    .reset_index(name="Ticket Count")
)
if not rated.empty:
    csat_distribution["Percentage of Rated Tickets"] = (
        csat_distribution["Ticket Count"] / len(rated) * 100
    ).round(2)
display(csat_distribution)

## 7. Customer satisfaction by ticket status

In [ ]:
csat_status = service_df[["Ticket Status", "Customer Satisfaction Rating"]].copy()
csat_status["Customer Satisfaction Rating"] = pd.to_numeric(
    csat_status["Customer Satisfaction Rating"], errors="coerce"
)

status_csat_summary = (
    csat_status.groupby("Ticket Status", dropna=False)
    .agg(
        Total_Tickets=("Customer Satisfaction Rating", "size"),
        Tickets_With_CSAT=("Customer Satisfaction Rating", "count"),
        Average_CSAT=("Customer Satisfaction Rating", "mean"),
        Median_CSAT=("Customer Satisfaction Rating", "median")
    )
    .reset_index()
)
status_csat_summary[["Average_CSAT", "Median_CSAT"]] = (
    status_csat_summary[["Average_CSAT", "Median_CSAT"]].round(2)
)
display(status_csat_summary)

# Plot only statuses with available CSAT ratings.
plot_summary = status_csat_summary.dropna(subset=["Average_CSAT"])
if not plot_summary.empty:
    plt.figure(figsize=(8, 5))
    bars = plt.bar(
        plot_summary["Ticket Status"].astype(str),
        plot_summary["Average_CSAT"]
    )
    plt.bar_label(
        bars,
        labels=[
            f"{v:.2f} (n={n})"
            for v, n in zip(
                plot_summary["Average_CSAT"],
                plot_summary["Tickets_With_CSAT"]
            )
        ],
        padding=3
    )
    plt.title("Average CSAT by Ticket Status")
    plt.xlabel("Ticket Status")
    plt.ylabel("Average CSAT (1–5)")
    plt.ylim(0, 5.5)
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()

## 8. Service-performance findings and interpretation

### Observed results

Complete this section after running the notebook and reviewing the generated tables:

- Report the number of records with each timestamp.
- Report how many records have both timestamps.
- Report negative timestamp sequences separately.
- Report median and P90 only for valid, non-negative intervals.
- Report CSAT coverage, average, median, and rating distribution.

### Interpretation rules

- The interval calculated here is specifically **time between first-response and resolution timestamps**.
- It must not be described as first-response duration from ticket creation or total resolution duration.
- A negative interval indicates a timestamp-ordering issue that requires investigation.
- No SLA breach rate can be calculated without a documented SLA target.
- CSAT comparisons are based only on records with available ratings.
- The dataset has no explicit escalation flag or ticket-created timestamp, so escalation-specific service comparisons and backlog ageing cannot be established.

## 9. Final limitations

1. No ticket-created timestamp is available.
2. No explicit escalation indicator is available.
3. No SLA target or unit definition is supplied.
4. First Response Time and Time to Resolution are timestamp fields; they are not numeric durations.
5. Missing CSAT ratings must not be interpreted as dissatisfied customers.
6. Observed associations do not establish causation.

**Conclusion:** This notebook describes the available timestamp coverage, validates timestamp ordering, calculates only the interval supported by the available fields, and summarises recorded customer satisfaction. It does not claim actual escalation rates, SLA breaches, or total resolution duration from ticket creation.